<a href="https://colab.research.google.com/github/Askaaarrr/NUSIF-PM-work/blob/main/portfolio_analytics.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Portfolio Analytics

A quantitative portfolio toolkit built from first principles: **data → returns → risk → optimization → VaR → factors → option pricing**, with visualizations that render on GitHub.

Everything runs on sample data out of the box. Replace the sample block in section 1 with your real portfolio file (`.csv` / `.xlsx` / `.dta`) to use your own data.

Each section has a short explanation above the code, and the code is commented so you can study it line by line.

_Author: Askar Shilderkhan_

## 0. Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.optimize import minimize   # for risk parity
from scipy.stats import norm          # for VaR and Black-Scholes
import statsmodels.api as sm          # for the factor regression

## 1. Load the portfolio

We want a **price table**: rows = dates, columns = tickers.
`load_portfolio` reads `.csv`, `.xlsx`, or `.dta` (Stata) by looking at the file extension.

In [ ]:
def load_portfolio(path):
    """Load a price table from .csv, .xlsx, or .dta. First column must be dates."""
    if path.endswith(".csv"):
        df = pd.read_csv(path, index_col=0, parse_dates=True)
    elif path.endswith((".xlsx", ".xls")):
        df = pd.read_excel(path, index_col=0, parse_dates=True)
    elif path.endswith(".dta"):
        df = pd.read_stata(path)
    else:
        raise ValueError("Use .csv, .xlsx, or .dta")
    return df.sort_index()

**Sample data** so the notebook runs immediately.
To use your own portfolio, comment this out and write:
`prices = load_portfolio("data/your_file.csv")`

In [ ]:
rng = np.random.default_rng(42)                 # fixed seed = reproducible
tickers = ["ASML", "NVDA", "MSFT", "AAPL", "COP"]
dates = pd.bdate_range("2023-01-01", periods=120, freq="W")

# simulate correlated-ish weekly prices via a random walk in log space
steps = rng.normal(0, 0.02, size=(len(dates), len(tickers)))
prices = pd.DataFrame(100 * np.exp(np.cumsum(steps, axis=0)),
                      index=dates, columns=tickers)
prices.head()

## 2. Returns

Convert prices to **weekly simple returns**: `r_t = P_t / P_{t-1} - 1`.
`.pct_change()` does exactly this; the first row is `NaN` (no prior price), so we drop it.

In [ ]:
returns = prices.pct_change().dropna()
returns.head()

## 3. Covariance & correlation

The **covariance matrix** `Sigma` is the central input for portfolio risk and optimization.
The **correlation matrix** is the same information rescaled to [-1, 1] and is easier to read.

We keep two forms: `cov` / `corr` as pandas (labeled) and `Sigma` as a plain NumPy array for the linear algebra below.

In [ ]:
cov  = returns.cov()
corr = returns.corr()
Sigma = cov.values          # NumPy array for matrix math
mu    = returns.mean().values   # mean weekly return per asset
n     = len(tickers)
corr.round(2)

## 4. Visualization — correlation heatmap

Shows how assets move together. Saved as PNG so it can go in a README, and it renders inline on GitHub.

In [ ]:
fig, ax = plt.subplots(figsize=(6, 5))
im = ax.imshow(corr, cmap="RdBu_r", vmin=-1, vmax=1)
ax.set_xticks(range(n)); ax.set_xticklabels(corr.columns, rotation=45, ha="right")
ax.set_yticks(range(n)); ax.set_yticklabels(corr.index)
fig.colorbar(im, label="correlation")
ax.set_title("Portfolio correlation matrix")
plt.tight_layout(); plt.savefig("correlation_heatmap.png", dpi=120); plt.show()

## 5. Portfolio volatility

For weights `w`, portfolio variance is `wᵀ Σ w` and volatility is its square root.
`w @ Sigma @ w` is the matrix form of that quadratic.

In [ ]:
def portfolio_volatility(w, Sigma):
    """Portfolio volatility = sqrt(wᵀ Σ w)."""
    w = np.asarray(w)
    return float(np.sqrt(w @ Sigma @ w))

# equal-weight portfolio as a reference point
w_eq = np.repeat(1 / n, n)
print("Equal-weight weekly volatility:", round(portfolio_volatility(w_eq, Sigma), 5))

## 6. Euler risk contributions

Total volatility can be split into a piece from each asset (they sum exactly to `σ_p`).
- **Marginal** contribution: `MCR = Σw / σ_p`
- **Component** contribution: `CCR = w ⊙ MCR`, and `Σ CCR = σ_p`
- **Percent** contribution: `CCR / σ_p`

This is how you see *concentration* — which names actually drive the risk, regardless of their weight.

In [ ]:
def risk_contributions(w, Sigma):
    """Return (component contributions, percent contributions) via Euler's theorem."""
    w = np.asarray(w)
    s = np.sqrt(w @ Sigma @ w)     # portfolio vol
    mcr = (Sigma @ w) / s          # marginal contribution to risk
    ccr = w * mcr                  # component contribution (sums to s)
    return ccr, ccr / s            # absolute, percent

ccr, pct = risk_contributions(w_eq, Sigma)
pd.DataFrame({"weight": w_eq, "risk %": pct}, index=tickers).round(3)

### Visualization — risk contribution vs weight

In [ ]:
x = np.arange(n)
fig, ax = plt.subplots(figsize=(7, 4))
ax.bar(x - 0.2, w_eq, width=0.4, label="weight")
ax.bar(x + 0.2, pct,  width=0.4, label="risk contribution")
ax.set_xticks(x); ax.set_xticklabels(tickers)
ax.set_ylabel("share of portfolio"); ax.set_title("Weight vs. risk contribution")
ax.legend(); plt.tight_layout(); plt.savefig("risk_contributions.png", dpi=120); plt.show()

## 7. Optimization

Three classic portfolios:

1. **Global Minimum Variance (GMV)** — lowest possible variance. Closed form:
   `w = Σ⁻¹ 1 / (1ᵀ Σ⁻¹ 1)`
2. **Tangency / max-Sharpe** — best risk-adjusted return given a risk-free rate `rf`. Closed form:
   `w ∝ Σ⁻¹ (μ − rf·1)`
3. **Risk parity** — every asset contributes *equal risk*. No closed form, so we solve numerically.

In [ ]:
inv  = np.linalg.inv(Sigma)
ones = np.ones(n)

# 1) GMV
w_gmv = inv @ ones / (ones @ inv @ ones)

# 2) Tangency (max Sharpe), weekly risk-free rate rf
rf = 0.0
excess = mu - rf
w_tan = inv @ excess / (ones @ inv @ excess)

# 3) Risk parity: minimize dispersion of component risk contributions,
#    subject to weights summing to 1 and being non-negative.
def rp_objective(w, Sigma):
    ccr, _ = risk_contributions(w, Sigma)
    return np.sum((ccr - ccr.mean())**2)

constraints = ({"type": "eq", "fun": lambda w: w.sum() - 1},)
bounds = tuple((0, 1) for _ in range(n))
w_rp = minimize(rp_objective, w_eq, args=(Sigma,),
                method="SLSQP", bounds=bounds, constraints=constraints).x

pd.DataFrame({"GMV": w_gmv, "Tangency": w_tan, "RiskParity": w_rp},
             index=tickers).round(3)

## 8. Efficient frontier

Every point is the *minimum variance* achievable for a target return.
Using `Σ⁻¹` we define scalars `A, B, C, D` and get a closed-form curve — no loops:

`σ²(m) = (A·m² − 2B·m + C) / D`,  where `m` is the target mean return.

In [ ]:
A = ones @ inv @ ones
B = ones @ inv @ mu
C = mu   @ inv @ mu
D = A * C - B**2

m_grid   = np.linspace(mu.min(), mu.max(), 100)   # array of target returns
sig_grid = np.sqrt((A * m_grid**2 - 2 * B * m_grid + C) / D)

# GMV point in (risk, return) space
m_gmv, sig_gmv = B / A, np.sqrt(1 / A)
asset_sig = np.sqrt(np.diag(Sigma))               # each asset on its own

fig, ax = plt.subplots(figsize=(7, 5))
ax.plot(sig_grid, m_grid, label="efficient frontier")
ax.scatter(asset_sig, mu, marker="o", label="individual assets")
ax.scatter(sig_gmv, m_gmv, marker="*", s=200, label="GMV")
for i, t in enumerate(tickers):
    ax.annotate(t, (asset_sig[i], mu[i]), fontsize=8,
                xytext=(4, 4), textcoords="offset points")
ax.set_xlabel("weekly volatility"); ax.set_ylabel("weekly return")
ax.set_title("Efficient frontier"); ax.legend()
plt.tight_layout(); plt.savefig("efficient_frontier.png", dpi=120); plt.show()

## 9. Value-at-Risk (VaR)

The loss not exceeded with probability `1 − α` (here `α = 5%`). Three standard estimates:

- **Historical** — empirical 5th percentile of realized portfolio returns.
- **Parametric (Gaussian)** — `−(μ_p + z_α·σ_p)`, assuming normality.
- **Monte Carlo** — simulate many draws from a multivariate normal fit to the data, then take the percentile.

In [ ]:
alpha = 0.05
w = w_eq                                   # evaluate the equal-weight book

port_ret  = returns.values @ w             # historical portfolio returns
var_hist  = -np.percentile(port_ret, 100 * alpha)

mu_p, sd_p = port_ret.mean(), port_ret.std(ddof=1)
var_param  = -(mu_p + norm.ppf(alpha) * sd_p)

sims      = rng.multivariate_normal(mu, Sigma, size=100_000)   # vectorized, no loop
var_mc    = -np.percentile(sims @ w, 100 * alpha)

pd.Series({"Historical": var_hist, "Parametric": var_param, "MonteCarlo": var_mc}).round(4)

## 10. Fama-French factor regression

Regress an asset's excess return on factor returns to get its **exposures (betas)** and **alpha**:
`r − rf = α + β_MKT·MKT + β_SMB·SMB + β_HML·HML + ε`

Here factors are simulated so the cell runs. For real work, download Fama-French factors
(Ken French data library) and align them to your return dates.

In [ ]:
# synthetic factors (replace with real Fama-French data)
factors = pd.DataFrame(rng.normal(0, 0.01, size=(len(returns), 3)),
                       index=returns.index, columns=["MKT", "SMB", "HML"])

y = returns["ASML"] - rf          # asset excess return
X = sm.add_constant(factors)      # adds the alpha (intercept) term
model = sm.OLS(y, X).fit()
print(model.summary().tables[1])  # coefficients, t-stats, p-values

## 11. Black-Scholes option pricing

Closed-form price of a European option:
`d1 = [ln(S/K) + (r + ½σ²)T] / (σ√T)`, `d2 = d1 − σ√T`,
call `= S·N(d1) − K·e^{−rT}·N(d2)`, put by symmetry.

In [ ]:
def black_scholes(S, K, T, r, sigma, kind="call"):
    """European option price. S spot, K strike, T years, r rate, sigma vol."""
    d1 = (np.log(S / K) + (r + 0.5 * sigma**2) * T) / (sigma * np.sqrt(T))
    d2 = d1 - sigma * np.sqrt(T)
    if kind == "call":
        return S * norm.cdf(d1) - K * np.exp(-r * T) * norm.cdf(d2)
    return K * np.exp(-r * T) * norm.cdf(-d2) - S * norm.cdf(-d1)

call = black_scholes(100, 100, 1, 0.05, 0.20, "call")
put  = black_scholes(100, 100, 1, 0.05, 0.20, "put")
print(f"Call: {call:.4f}   Put: {put:.4f}")

## Publish

**From Google Colab:** `File → Save a copy in GitHub` → choose or create a repo → commit.
The notebook and all three charts render on GitHub automatically.

**What this covers:** returns, covariance, portfolio volatility, Euler risk decomposition,
GMV / tangency / risk-parity optimization, the efficient frontier, three VaR methods,
a factor regression, and Black-Scholes — the core of a quant portfolio workflow.